In [ ]:
from pathlib import Path

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    ArrayType,
    DoubleType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

spark = SparkSession.builder.appName("nyc311").getOrCreate()
DATA_DIR = Path("/opt/airflow/data")

RAW_DIR = DATA_DIR / "raw"

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 14:58:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
NYC311_SCHEMA = StructType([
    # Identifiers and timestamps
    StructField("unique_key", StringType(), True),
    StructField("created_date", TimestampType(), True),
    StructField("closed_date", TimestampType(), True),

    # Complaint information
    StructField("agency", StringType(), True),
    StructField("agency_name", StringType(), True),
    StructField("complaint_type", StringType(), True),
    StructField("descriptor", StringType(), True),
    StructField("descriptor_2", StringType(), True),
    StructField("location_type", StringType(), True),

    # Address information
    StructField("incident_zip", StringType(), True),
    StructField("incident_address", StringType(), True),
    StructField("street_name", StringType(), True),
    StructField("cross_street_1", StringType(), True),
    StructField("cross_street_2", StringType(), True),
    StructField("intersection_street_1", StringType(), True),
    StructField("intersection_street_2", StringType(), True),
    StructField("address_type", StringType(), True),
    StructField("city", StringType(), True),
    StructField("landmark", StringType(), True),
    StructField("facility_type", StringType(), True),

    # Request status and resolution
    StructField("status", StringType(), True),
    StructField("due_date", TimestampType(), True),
    StructField("resolution_description", StringType(), True),
    StructField("resolution_action_updated_date", TimestampType(), True),

    # Administrative geography
    StructField("community_board", StringType(), True),
    StructField("council_district", StringType(), True),
    StructField("police_precinct", StringType(), True),
    StructField("bbl", StringType(), True),
    StructField("borough", StringType(), True),

    # State plane coordinates
    StructField("x_coordinate_state_plane", DoubleType(), True),
    StructField("y_coordinate_state_plane", DoubleType(), True),

    # Request channel and location details
    StructField("open_data_channel_type", StringType(), True),
    StructField("park_facility_name", StringType(), True),
    StructField("park_borough", StringType(), True),
    StructField("vehicle_type", StringType(), True),
    StructField("taxi_company_borough", StringType(), True),
    StructField("taxi_pick_up_location", StringType(), True),
    StructField("bridge_highway_name", StringType(), True),
    StructField("bridge_highway_direction", StringType(), True),
    StructField("road_ramp", StringType(), True),
    StructField("bridge_highway_segment", StringType(), True),

    # Geographic coordinates
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),

    # GeoJSON Point (SODA 3.0)
    StructField(
        "location",
        StructType([
            StructField("type", StringType(), True),
            StructField(
                "coordinates",
                ArrayType(DoubleType(), True),
                True
            ),
        ]),
        True
    ),
])


In [3]:
spark_df = spark.read.schema(NYC311_SCHEMA).json(str(RAW_DIR / "2026-11-10.jsonl.gz"))
spark_df.printSchema()

root
 |-- unique_key: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- closed_date: timestamp (nullable = true)
 |-- agency: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- descriptor: string (nullable = true)
 |-- descriptor_2: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- incident_zip: string (nullable = true)
 |-- incident_address: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- cross_street_1: string (nullable = true)
 |-- cross_street_2: string (nullable = true)
 |-- intersection_street_1: string (nullable = true)
 |-- intersection_street_2: string (nullable = true)
 |-- address_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- landmark: string (nullable = true)
 |-- facility_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- due_date: timestamp (nullable = true)
 |-- resolution_descrip

In [5]:
n_rows = spark_df.count()

profile = spark_df.agg(*[
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in spark_df.columns
])

null_profile = (
    profile
    .select(
        F.explode(
            F.array(*[
                F.struct(
                    F.lit(c).alias("column"),
                    F.col(c).alias("nulls"),
                )
                for c in spark_df.columns
            ])
        ).alias("x")
    )
    .select(
        "x.column",
        "x.nulls",
        F.round(F.col("x.nulls") / n_rows * 100, 2).alias("null_pct"),
    )
    .orderBy(F.desc("null_pct"))
)

null_profile.show(50, truncate=False)

+------------------------------+-----+--------+
|column                        |nulls|null_pct|
+------------------------------+-----+--------+
|x_coordinate_state_plane      |12025|100.0   |
|y_coordinate_state_plane      |12025|100.0   |
|latitude                      |12025|100.0   |
|longitude                     |12025|100.0   |
|taxi_company_borough          |12012|99.89   |
|due_date                      |11997|99.77   |
|road_ramp                     |11983|99.65   |
|bridge_highway_direction      |11975|99.58   |
|bridge_highway_name           |11936|99.26   |
|bridge_highway_segment        |11936|99.26   |
|taxi_pick_up_location         |11919|99.12   |
|facility_type                 |11745|97.67   |
|vehicle_type                  |11559|96.12   |
|closed_date                   |6174 |51.34   |
|descriptor_2                  |5357 |44.55   |
|landmark                      |5144 |42.78   |
|cross_street_2                |4361 |36.27   |
|cross_street_1                |4356 |36

In [6]:
columns = null_profile.filter(null_profile["null_pct"] <= 10).select("column").collect()

for c in columns:
    print(c["column"])
    (
        spark_df.groupBy(c["column"])
        .count()
        .orderBy(F.desc("count"))
        .show(100, truncate=False)
    )
    print("===========================================================\n")

location_type
+-----------------------------+-----+
|location_type                |count|
+-----------------------------+-----+
|Street/Sidewalk              |3854 |
|RESIDENTIAL BUILDING         |3608 |
|Street                       |987  |
|Residential Building/House   |975  |
|NULL                         |825  |
|Sidewalk                     |561  |
|Store/Commercial             |114  |
|Building/Construction Site   |97   |
|Residential Building         |90   |
|Park                         |88   |
|N/A                          |83   |
|Sewer                        |79   |
|Business                     |60   |
|Restaurant/Bar/Deli/Bakery   |59   |
|Subway                       |57   |
|Residential                  |44   |
|3+ Family Apt. Building      |36   |
|Commercial/Mixed Use Building|35   |
|3+ Family Apartment Building |34   |
|Club/Bar/Restaurant          |32   |
|Park/Playground              |29   |
|Highway                      |27   |
|Other                        |20   